Døme på korleis ein kan lese ut ein organisasjonsstruktur fra NVA

Programmet trenger å vite kva for ein institusjon du vil skrive ut til fil
og kva språk du vil ha listet ut (nb, en, nn).

In [3]:
import ipywidgets as widgets

instkode = widgets.Text(
    value="185",                  # whatever default makes sense
    description="Org code:",
    placeholder="e.g. 185",
)

lang = widgets.Dropdown(
    options=[("English", "en"), ("Bokmål", "nb"), ("Nynorsk", "nn")],
    value="nb",
    description="Language:",
)

export_format = widgets.Dropdown(
    options=[("Display only (no file)", "display"),
             ("CSV", "csv"),
             ("XLSX (Norwegian formatting)", "xlsx")],
    value="xlsx",
    description="Export:",
)

widgets.VBox([instkode, lang, export_format])   # set all three, then run the processing cell

In [5]:
# pip install requests pandas rdflib ipywidgets openpyxl
#   (openpyxl is the pandas engine for .xlsx; no pyld pin needed on 3.14)
import json, requests, pandas as pd
from rdflib import Graph
from IPython.display import display

URL = "https://api.nva.unit.no/cristin/organization/" + instkode.value
doc = requests.get(URL, headers={"Accept": "application/json"}, timeout=30).json()

LANG = lang.value                 # "en" / "nb" / "nn"
COLS = ["uri", LANG]

QUERY = """
PREFIX pub: <https://nva.sikt.no/ontology/publication#>
SELECT ?uri (SAMPLE(?enLabel) AS ?en) (SAMPLE(?nbLabel) AS ?nb) (SAMPLE(?nnLabel) AS ?nn)
WHERE {
  ?uri a pub:Organization .
  { ?uri pub:label ?enLabel . FILTER(lang(?enLabel) = "en") }
  UNION { ?uri pub:label ?nbLabel . FILTER(lang(?nbLabel) = "nb") }
  UNION { ?uri pub:label ?nnLabel . FILTER(lang(?nnLabel) = "nn") }
}
GROUP BY ?uri
"""

def table_sparql(doc):
    g = Graph()
    g.parse(data=json.dumps(doc), format="json-ld")   # also fetches the remote @context
    rows = [
        {c: (str(r.get(c)) if r.get(c) is not None else None) for c in COLS}
        for r in g.query(QUERY)
    ]
    return pd.DataFrame(rows).reindex(columns=COLS)

def write_xlsx_no(frame, path):
    """Write an .xlsx, forcing every data cell to text so org codes/ids
    aren't reinterpreted by Excel (leading zeros, dotted codes, etc.)."""
    with pd.ExcelWriter(path, engine="openpyxl") as xl:
        frame.to_excel(xl, index=False, sheet_name="Sheet1")
        ws = xl.sheets["Sheet1"]
        for row in ws.iter_rows(min_row=2):       # skip the header row
            for cell in row:
                cell.number_format = "@"          # text format
        for col in ws.columns:                    # rough auto-width
            width = max((len(str(c.value)) for c in col if c.value is not None), default=0) + 2
            ws.column_dimensions[col[0].column_letter].width = width

df = table_sparql(doc)
missing = df[LANG].isna().sum()
df[LANG] = df[LANG].fillna("")        # keep every org; gaps stay visible as blanks
count = len(df)

if count == 0:
    print("No data – is the instkode correct?")
else:
    df = df.sort_values("uri")
    langcol = df.columns[1]

    uri_df = df[["uri", langcol]].copy()
    id_df = df.assign(id=df["uri"].str.extract(r"organization/(.+)"))[["id", langcol]]

    fmt = export_format.value
    if fmt == "display":
        print(count, "rows;", missing, "with no label in", langcol, "— not saved")
        display(uri_df, id_df)
    elif fmt == "csv":
        uri_df.to_csv(f"uri_{instkode.value}_{langcol}.csv", index=False, encoding="utf-8-sig")
        id_df.to_csv(f"id_{instkode.value}_{langcol}.csv", index=False, encoding="utf-8-sig")
        print(count, "rows written to two CSV files;", missing, "with no label in", langcol)
    elif fmt == "xlsx":
        write_xlsx_no(uri_df, f"uri_{instkode.value}_{langcol}.xlsx")
        write_xlsx_no(id_df, f"id_{instkode.value}_{langcol}.xlsx")
        print(count, "rows written to two XLSX files;", missing, "with no label in", langcol)

2 rows written to two XLSX files; 0 with no label in nb
